# 06 Evaluation Report — итог по matching и fusion

Это короткий отчёт по текущему research-прогону: качество threshold benchmark, выбранный fusion-run, опасные false merge и результат family/pack-группировки.


## Блок кода 1. Подготовка окружения


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "research" / "dedup").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


## Блок кода 2. Загрузка готовых артефактов


In [ ]:
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"

LABELING_PATH = DATA_DIR / "labeling_sauces.csv"
SUMMARY_PATH = REPORTS_DIR / "binary_threshold_summary.csv"
PREDICTIONS_PATH = REPORTS_DIR / "binary_threshold_predictions.csv"
COMPONENTS_PATH = DATA_DIR / "fusion_components_sauces.csv"
PAIR_EVAL_PATH = DATA_DIR / "fusion_pair_eval_sauces.csv"

missing = [path for path in [SUMMARY_PATH, PREDICTIONS_PATH, COMPONENTS_PATH, PAIR_EVAL_PATH] if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required artifacts: " + ", ".join(str(path) for path in missing))

labels = pd.read_csv(LABELING_PATH) if LABELING_PATH.exists() else pd.DataFrame()
threshold_summary = pd.read_csv(SUMMARY_PATH)
threshold_predictions = pd.read_csv(PREDICTIONS_PATH)
components = pd.read_csv(COMPONENTS_PATH)
pair_eval = pd.read_csv(PAIR_EVAL_PATH)

print(f"Labels: {len(labels)} rows")
print(f"Threshold summary: {len(threshold_summary)} rows")
print(f"Threshold predictions: {len(threshold_predictions)} rows")
print(f"Fusion components: {len(components)} rows")
print(f"Fusion pair eval: {len(pair_eval)} rows")


## Блок кода 3. Состояние gold-set


In [ ]:
if labels.empty or "label" not in labels.columns:
    print("labeling_sauces.csv не найден или в нём нет label; пропускаем разбор разметки")
else:
    label_counts = labels["label"].fillna("<empty>").astype(str).value_counts(dropna=False).rename_axis("label").reset_index(name="pairs")
    display(label_counts)


## Блок кода 4. Сравнение methods на test

`test` здесь только для чтения результата. Выбор threshold уже был сделан на `dev` в `03`.


In [ ]:
test_summary = threshold_summary[threshold_summary["split"].astype(str).eq("test")].copy()
if test_summary.empty:
    test_summary = threshold_summary.copy()

ranking_cols = [
    "method",
    "threshold_strategy",
    "threshold_same",
    "precision",
    "recall",
    "f1",
    "false_merge_count",
    "false_split_count",
    "cost",
    "weighted_f1",
    "weighted_total_cost",
    "weight_source",
]
ranking_cols = [column for column in ranking_cols if column in test_summary.columns]
ranked = test_summary.sort_values(
    ["cost", "false_merge_count", "false_split_count", "f1"],
    ascending=[True, True, True, False],
)
display(ranked[ranking_cols].head(20))


## Блок кода 5. Выбранный fusion-run и graph quality


In [ ]:
def _as_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes"})


def _binary_link_report(frame: pd.DataFrame, *, true_col: str, pred_col: str, scope: str) -> dict[str, object]:
    true_link = _as_bool(frame[true_col])
    pred_link = _as_bool(frame[pred_col])
    tp = int((true_link & pred_link).sum())
    fp = int((~true_link & pred_link).sum())
    fn = int((true_link & ~pred_link).sum())
    tn = int((~true_link & ~pred_link).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "scope": scope,
        "pairs": len(frame),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "false_links": fp,
        "missed_links": fn,
    }


fusion_identity_cols = ["fusion_method", "fusion_threshold_strategy", "fusion_threshold_same"]
if set(fusion_identity_cols).issubset(pair_eval.columns):
    display(pair_eval[fusion_identity_cols].drop_duplicates())

heldout_pairs = pair_eval[pair_eval["split"].astype(str).eq("test")].copy() if "split" in pair_eval.columns else pair_eval.copy()
if heldout_pairs.empty:
    heldout_pairs = pair_eval.copy()

graph_quality = pd.DataFrame(
    [
        _binary_link_report(heldout_pairs, true_col="true_same_family", pred_col="pred_same_family", scope="family"),
        _binary_link_report(heldout_pairs, true_col="true_same_pack", pred_col="pred_same_pack", scope="pack"),
    ]
)
display(graph_quality)


## Блок кода 6. Опасные false merge

Это пары, где target говорит `different_product`, но выбранный threshold/fusion решил связать их как один базовый товар.


In [ ]:
selected_method = None
selected_strategy = None
if {"fusion_method", "fusion_threshold_strategy"}.issubset(pair_eval.columns) and not pair_eval.empty:
    selected_method = str(pair_eval["fusion_method"].dropna().iloc[0])
    selected_strategy = str(pair_eval["fusion_threshold_strategy"].dropna().iloc[0])

danger = threshold_predictions.copy()
if selected_method is not None:
    danger = danger[danger["method"].astype(str).eq(selected_method)]
if selected_strategy is not None:
    danger = danger[danger["threshold_strategy"].astype(str).eq(selected_strategy)]
if "false_merge" in danger.columns:
    danger = danger[_as_bool(danger["false_merge"])]
else:
    danger = danger.iloc[0:0]

danger_cols = [
    "split",
    "score",
    "threshold_same",
    "title_a",
    "title_b",
    "brand_a",
    "brand_b",
    "unit_amount_a",
    "unit_amount_b",
    "total_amount_a",
    "total_amount_b",
    "multipack_count_a",
    "multipack_count_b",
    "pair_weight",
]
danger_cols = [column for column in danger_cols if column in danger.columns]
display(danger[danger_cols].head(20))


## Блок кода 7. Итог человеческим языком


In [ ]:
conclusions = []
if not ranked.empty:
    best = ranked.iloc[0]
    conclusions.append(
        f"По test-таблице лучший верхний кандидат: {best['method']} / {best['threshold_strategy']} с cost={best['cost']}."
    )
if not graph_quality.empty:
    family = graph_quality[graph_quality["scope"].eq("family")].iloc[0]
    pack = graph_quality[graph_quality["scope"].eq("pack")].iloc[0]
    conclusions.append(
        f"Family graph: precision={family['precision']:.3f}, recall={family['recall']:.3f}, false_links={int(family['false_links'])}."
    )
    conclusions.append(
        f"Pack graph: precision={pack['precision']:.3f}, recall={pack['recall']:.3f}, false_links={int(pack['false_links'])}."
    )
conclusions.append(
    "Разные фасовки не являются отдельным ML-классом: они выделяются после matching через unit/total/multipack signature."
)
display(pd.DataFrame({"conclusion": conclusions}))


## Что делать после отчёта

Если false merge всё ещё много, следующий шаг — улучшать scorer/reranker на hard negatives. Если family уже достаточно чистая, можно отдельно улучшать pack signature и готовить перенос выбранного подхода из research в production pipeline.
